# Credit Default Prediction — Part 3: Modeling & Evaluation

**Goal:** Train and compare three classification models. Evaluate using metrics appropriate for imbalanced data in a high-stakes lending context. Select a final model and calibrate its probabilities for business use.

**Why not accuracy?** With a ~6.7% default rate, a model that predicts "no default" for every borrower achieves 93.3% accuracy — and is completely useless. We use AUC-ROC and precision-recall curves instead.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    classification_report, confusion_matrix
)
import joblib

plt.style.use('seaborn-v0_8-whitegrid')

X_train = pd.read_csv('../data/X_train.csv')
X_test  = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/y_test.csv').squeeze()

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 1. Model Training

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=6, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight='balanced',
        n_jobs=-1, random_state=42
    )
}

results = {}
for name, model in models.items():
    X_tr = X_train_scaled if name == 'Logistic Regression' else X_train
    X_te = X_test_scaled  if name == 'Logistic Regression' else X_test
    model.fit(X_tr, y_train)
    proba = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_test, proba)
    ap  = average_precision_score(y_test, proba)
    results[name] = {'model': model, 'proba': proba, 'auc': auc, 'ap': ap}
    print(f'{name:25s} | AUC-ROC: {auc:.4f} | Avg Precision: {ap:.4f}')

## 2. ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#3498db', '#e67e22', '#2ecc71']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate (Approve bad borrower)')
ax.set_ylabel('True Positive Rate (Catch default)')
ax.set_title('ROC Curve — Model Comparison', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/roc_comparison.png', dpi=150)
plt.show()

## 3. Feature Importance (Random Forest)

In [ ]:
rf = results['Random Forest']['model']
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
importances.plot(kind='barh', ax=ax, color='#3498db', edgecolor='white')
ax.set_title('Random Forest — Feature Importance', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../outputs/figures/feature_importance.png', dpi=150)
plt.show()

## 4. Probability Calibration

For a lending model, we need well-calibrated probabilities — when the model says 15% default risk, roughly 15% of those borrowers should actually default. Uncalibrated probabilities are misleading for threshold-based approval decisions.

In [ ]:
rf_calibrated = CalibratedClassifierCV(
    RandomForestClassifier(n_estimators=200, max_depth=10,
                           class_weight='balanced', n_jobs=-1, random_state=42),
    method='isotonic', cv=5
)
rf_calibrated.fit(X_train, y_train)
proba_cal = rf_calibrated.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(6, 5))
for proba, label, color in [
    (results['Random Forest']['proba'], 'Uncalibrated RF', '#e74c3c'),
    (proba_cal, 'Calibrated RF', '#2ecc71')
]:
    fraction_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10)
    ax.plot(mean_pred, fraction_pos, marker='o', label=label, color=color)

ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives (Actual Default Rate)')
ax.set_title('Calibration Curve', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/calibration_curve.png', dpi=150)
plt.show()

# Save final model
joblib.dump(rf_calibrated, '../outputs/credit_default_model.pkl')
joblib.dump(proba_cal, '../outputs/test_probabilities.pkl')
print('Calibrated model saved.')